# Hybrid Search Experiments

**BM25, Chunked Semantic Search, and Retrieval Fusion Experiments**

This notebook documents the development and evaluation of hybrid retrieval strategies for Origin, a system designed to identify historical graduation projects related to newly proposed project ideas.

The experiments investigate the combination of BM25 keyword-based retrieval and multilingual Semantic Search using Reciprocal Rank Fusion (RRF).

### Experimental Scope

- Improve Arabic lexical retrieval through Arabic-to-English query translation.
- Integrate token-based chunking into multilingual Semantic Search.
- Evaluate Standard Hybrid Search using RRF.
- Analyze Arabic retrieval failures and their underlying causes.
- Experiment with Weighted RRF and different BM25 fusion weights.
- Compare retrieval performance across Arabic and English development queries.
- Investigate a language-aware retrieval strategy based on experimental results.

### Evaluation Approach

The experiments use 40 development queries (20 Arabic and 20 English). Reference Hit@5 and MRR@5 are calculated using 26 queries with human-selected reference project IDs.

The findings support the selection of Origin's language-aware retrieval strategy, whose clean implementation is provided in `07_Language_Aware_Retrieval.ipynb`.

The reserved final test set is not used for model selection or parameter tuning.


## 1. Setup and Dependencies

Install the libraries required for BM25, semantic embeddings, and hybrid retrieval.

In [ ]:
%pip install -q sentence-transformers rank-bm25

In [ ]:
import pandas as pd
import numpy as np
import re
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

print("Libraries imported successfully!")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Libraries imported successfully!
Device: cuda


## 2. Load and Validate Dataset

Load the cleaned graduation projects dataset from GitHub and verify its structure before hybrid retrieval.

In [ ]:
# Load dataset
url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/processed/projects_clean.csv"
)

df = pd.read_csv(url, encoding="utf-8-sig")

# Standardize columns
df.columns = df.columns.str.strip().str.lower()
df["department"] = df["department"].str.strip()

# Validate dataset
assert df.shape == (103, 7)
assert df["project_id"].is_unique
assert df[["title", "abstract"]].notna().all().all()
assert df[["title", "abstract"]].apply(
    lambda col: col.astype(str).str.strip().ne("").all()
).all()

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Departments:", df["department"].nunique())

display(df.head(3))

Dataset loaded successfully!
Shape: (103, 7)
Departments: 6


,project_id,title,abstract,department,year,keywords,source_page
0,P001,Automatic Plant Watering and Tracking System,Many people struggle with watering their plant...,Computer Science,2024 – 2025,"Moisturem Air humidity, AWTS system, Waterin, ...",5
1,P002,علامة CourseAssist: CLO Performance Assistant ...,The rapid evolution of educational technology ...,Computer Science,2024 – 2025,"Program Learning Outcomes, Course Learning Out...",6
2,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",Computer Science,2024 – 2025,"Arabic Sign Language, Sign Language Gestures i...",7



## 3. Text Preparation and BM25 Preprocessing

This section combines project titles and abstracts and applies consistent lexical preprocessing for BM25.

The preprocessing includes Unicode normalization, lowercasing, tokenization, and English stop-word filtering.

The original project text remains unchanged for semantic retrieval. Arabic-to-English translation is performed later at query time for the BM25 component.


In [ ]:

# ============================================================
# 3. TEXT PREPARATION AND BM25 PREPROCESSING
# ============================================================

import re
import unicodedata
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

hybrid_df = df.copy()

# Original text for semantic retrieval
hybrid_df["text"] = (
    hybrid_df["title"].astype(str).str.strip()
    + ". "
    + hybrid_df["abstract"].astype(str).str.strip()
)

hybrid_df["text"] = (
    hybrid_df["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

project_texts = hybrid_df["text"].tolist()

# Consistent BM25 preprocessing
def tokenize(text):

    text = unicodedata.normalize(
        "NFKC", str(text)
    ).casefold()

    tokens = re.findall(r"(?u)\b\w+\b", text)

    tokens = [
        token for token in tokens
        if len(token) >= 2
        and token not in ENGLISH_STOP_WORDS
    ]

    return tokens


# Build tokenized historical corpus
tokenized_corpus = [
    tokenize(text)
    for text in project_texts
]

assert len(tokenized_corpus) == 103
assert all(len(tokens) > 0 for tokens in tokenized_corpus)

print("BM25 PREPROCESSING COMPLETED")
print("=" * 55)
print("Total projects:", len(project_texts))
print("Tokenized documents:", len(tokenized_corpus))
print("Sample tokens:", tokenized_corpus[0][:20])


BM25 PREPROCESSING COMPLETED
Total projects: 103
Tokenized documents: 103
Sample tokens: ['automatic', 'plant', 'watering', 'tracking', 'people', 'struggle', 'watering', 'plants', 'regularly', 'usually', 'leads', 'plants', 'unhealthy', 'dying', 'address', 'common', 'issue', 'important', 'monitor', 'basic']


## 4. BM25 Retrieval Model

Build a BM25 index using the tokenized graduation project descriptions.

In [ ]:
# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created successfully!")
print("Indexed projects:", len(tokenized_corpus))

BM25 index created successfully!
Indexed projects: 103



## 4.1 Arabic-to-English Query Translation for BM25

The historical project abstracts are primarily written in English. Therefore, Arabic user queries may have little or no lexical overlap with the BM25 index.

This section introduces a pretrained Arabic-to-English translation model to support cross-language lexical retrieval.

Arabic queries are translated before BM25 tokenization and scoring. English queries are processed directly without translation.

The multilingual semantic retrieval component continues to use the original user query. Translation is used only by BM25.


In [ ]:

%pip install -q sentencepiece sacremoses


In [ ]:

# ============================================================
# 4.1 ARABIC-TO-ENGLISH QUERY TRANSLATION FOR BM25
# Origin - Hybrid Search
# ============================================================

import re
import html
import torch

from functools import lru_cache
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------
# 1. Translation model configuration
# ------------------------------------------------------------

translation_model_name = "Helsinki-NLP/opus-mt-ar-en"

translation_device = (
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# 2. Load tokenizer and translation model
# ------------------------------------------------------------

translation_tokenizer = AutoTokenizer.from_pretrained(
    translation_model_name
)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    translation_model_name
).to(translation_device)

translation_model.eval()

# Maximum supported input tokens
translation_max_input_tokens = int(
    translation_model.config.max_position_embeddings
)

# ------------------------------------------------------------
# 3. Translation function
# ------------------------------------------------------------

@lru_cache(maxsize=256)
def translate_query_for_bm25(query_text):

    query_text = str(query_text).strip()

    if not query_text:
        raise ValueError("Query text cannot be empty.")

    # English-only queries do not require translation
    if not re.search(r"[\u0600-\u06FF]", query_text):
        return query_text

    # Tokenize Arabic query without truncation
    inputs = translation_tokenizer(
        query_text,
        return_tensors="pt",
        truncation=False
    )

    input_token_count = inputs["input_ids"].shape[1]

    # Prevent silent input truncation
    if input_token_count > translation_max_input_tokens:
        raise ValueError(
            f"Arabic query has {input_token_count} tokens, "
            f"exceeding the translation model limit "
            f"of {translation_max_input_tokens}."
        )

    # Move inputs to the selected device
    inputs = {
        key: value.to(translation_device)
        for key, value in inputs.items()
    }

    # Generate English translation
    with torch.inference_mode():

        output_ids = translation_model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=4
        )

    # Decode translated text
    translated = translation_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    if not translated:
        raise ValueError(
            "Translation returned empty text."
        )

    # --------------------------------------------------------
    # 4. Clean HTML entities and apostrophes
    # --------------------------------------------------------

    # Example: Parkinson &apos; s -> Parkinson's
    translated = re.sub(
        r"\s*&apos;\s*",
        "'",
        translated,
        flags=re.IGNORECASE
    )

    # Decode any remaining HTML entities
    translated = html.unescape(translated)

    # Normalize excessive whitespace
    translated = re.sub(
        r"\s+",
        " ",
        translated
    ).strip()

    return translated


# ------------------------------------------------------------
# 5. Model readiness check
# ------------------------------------------------------------

print("TRANSLATION MODEL LOADED SUCCESSFULLY!")
print("=" * 55)

print("Model:", translation_model_name)
print("Device:", translation_device)

print(
    "Maximum input tokens:",
    translation_max_input_tokens
)

print("Arabic-to-English translation is ready!")
print("English queries bypass translation.")
print("Translation is used by BM25 only.")


config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

TRANSLATION MODEL LOADED SUCCESSFULLY!
Model: Helsinki-NLP/opus-mt-ar-en
Device: cuda
Maximum input tokens: 512
Arabic-to-English translation is ready!
English queries bypass translation.
Translation is used by BM25 only.



## Translation Model Token Length Validation

This section verifies that Arabic development queries fit within the translation model's input sequence limit.

Token lengths are measured using the translation model's own tokenizer without truncation. The check helps ensure that query translation can be performed without silently discarding input text.


In [ ]:

# ============================================================
# TRANSLATION MODEL TOKEN LENGTH VALIDATION
# ============================================================

import pandas as pd

model_limit = int(
    translation_model.config.max_position_embeddings
)

EVAL_URL = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/evaluation/evaluation_queries.csv"
)

eval_df = pd.read_csv(EVAL_URL, encoding="utf-8-sig")

arabic_dev = eval_df[
    (eval_df["language"] == "ar")
    & (eval_df["split"] == "dev")
].copy()

def count_translation_tokens(row):

    text = (
        str(row["query_title"]).strip()
        + ". "
        + str(row["query_description"]).strip()
    )

    return len(
        translation_tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )

arabic_dev["token_count"] = arabic_dev.apply(
    count_translation_tokens,
    axis=1
)

arabic_dev["exceeds_limit"] = (
    arabic_dev["token_count"] > model_limit
)

print("TRANSLATION TOKEN DIAGNOSTIC")
print("=" * 55)
print("Model:", translation_model_name)
print("Model input limit:", model_limit)
print("Arabic development queries:", len(arabic_dev))
print("Average tokens:", round(
    arabic_dev["token_count"].mean(), 2
))
print("Maximum tokens:", arabic_dev["token_count"].max())
print("Queries exceeding limit:",
      arabic_dev["exceeds_limit"].sum())

display(
    arabic_dev[
        ["query_id", "token_count", "exceeds_limit"]
    ].sort_values("token_count", ascending=False)
)

assert not arabic_dev["exceeds_limit"].any(), (
    "Some queries exceed the translation input limit."
)

print("\nTranslation input length validation passed!")


TRANSLATION TOKEN DIAGNOSTIC
Model: Helsinki-NLP/opus-mt-ar-en
Model input limit: 512
Arabic development queries: 20
Average tokens: 106.85
Maximum tokens: 151
Queries exceeding limit: 0


,query_id,token_count,exceeds_limit
38,Q039,151,False
59,Q060,141,False
29,Q030,140,False
26,Q027,136,False
17,Q018,130,False
35,Q036,125,False
5,Q006,125,False
25,Q026,124,False
46,Q047,108,False
55,Q056,106,False



Translation input length validation passed!


## 5. BM25 Retrieval Function

Implement keyword-based retrieval using BM25 scores to rank the most relevant graduation projects.

In [ ]:

# ============================================================
# 5. BM25 RETRIEVAL WITH ARABIC QUERY TRANSLATION
# ============================================================

def bm25_search(title, description, top_k=5):

    if not title.strip() or not description.strip():
        raise ValueError(
            "Title and description must not be empty."
        )

    if not 1 <= top_k <= len(hybrid_df):
        raise ValueError("Invalid top_k.")

    original_query = (
        f"{title.strip()}. {description.strip()}"
    )

    # Translate Arabic queries; keep English unchanged
    lexical_query = translate_query_for_bm25(
        original_query
    )

    # Use the same preprocessing as the BM25 index
    query_tokens = tokenize(lexical_query)

    scores = bm25.get_scores(query_tokens)

    # Exclude projects without positive lexical evidence
    matched_indices = np.flatnonzero(scores > 0)

    result_columns = [
        "rank",
        "project_id",
        "title",
        "abstract",
        "department",
        "source_page",
        "bm25_score"
    ]

    if len(matched_indices) == 0:
        return pd.DataFrame(columns=result_columns)

    sorted_indices = matched_indices[
        np.argsort(
            -scores[matched_indices],
            kind="stable"
        )
    ][:top_k]

    results = hybrid_df.iloc[sorted_indices][
        [
            "project_id",
            "title",
            "abstract",
            "department",
            "source_page"
        ]
    ].copy()

    results.insert(
        0, "rank", range(1, len(results) + 1)
    )

    results["bm25_score"] = scores[sorted_indices]

    return results.reset_index(drop=True)


print("Translation-aware BM25 retrieval is ready!")


Translation-aware BM25 retrieval is ready!


In [ ]:

# ============================================================
# ARABIC QUERY TRANSLATION TEST
# ============================================================

test_arabic_title = (
    "الكشف المبكر عن مرض باركنسون باستخدام الذكاء الاصطناعي"
)

test_arabic_description = (
    "تطبيق يستخدم تعلم الآلة وبيانات أجهزة الاستشعار "
    "القابلة للارتداء للكشف المبكر عن مرض باركنسون "
    "ومراقبة أعراض المرض لدى المرضى."
)

original_query = (
    test_arabic_title + ". " + test_arabic_description
)

translated_query = translate_query_for_bm25(
    original_query
)

scores = bm25.get_scores(
    tokenize(translated_query)
)

print("ORIGINAL ARABIC QUERY:")
print(original_query)

print("\nTRANSLATED ENGLISH QUERY:")
print(translated_query)

print("\nBM25 MATCHING DIAGNOSTICS")
print("Total historical projects:", len(scores))
print("Positive BM25 matches:", int((scores > 0).sum()))
print("Zero BM25 scores:", int((scores == 0).sum()))

print("\nBM25 TOP RESULTS")
display(
    bm25_search(
        test_arabic_title,
        test_arabic_description,
        top_k=5
    )[["rank", "project_id", "title", "bm25_score"]]
)

# Hybrid can be tested after the semantic model
# and project embeddings have been generated.
if "project_embeddings" in globals():

    print("\nHYBRID TOP RESULTS")

    display(
        hybrid_search(
            test_arabic_title,
            test_arabic_description,
            top_k=5
        )[
            [
                "rank",
                "project_id",
                "title",
                "bm25_rank",
                "semantic_rank",
                "rrf_score",
                "retrieval_mode"
            ]
        ]
    )


[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ORIGINAL ARABIC QUERY:
الكشف المبكر عن مرض باركنسون باستخدام الذكاء الاصطناعي. تطبيق يستخدم تعلم الآلة وبيانات أجهزة الاستشعار القابلة للارتداء للكشف المبكر عن مرض باركنسون ومراقبة أعراض المرض لدى المرضى.

TRANSLATED ENGLISH QUERY:
Early detection of Parkinson's disease using artificial intelligence. Application uses machine learning and wearable sensor data for early detection of Parkinson's disease and surveillance of patients'symptoms.

BM25 MATCHING DIAGNOSTICS
Total historical projects: 103
Positive BM25 matches: 92
Zero BM25 scores: 11

BM25 TOP RESULTS


,rank,project_id,title,bm25_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,49.043277
1,2,P005,Thabat – ثبات,31.579895
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,18.902648
3,4,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,15.595790
4,5,P026,Potato Guard,14.488951


## 6. BM25 Retrieval Testing

Test BM25 retrieval using the same English project idea evaluated in the Semantic Search notebook.

In [ ]:
# Test English query
test_title = "AI-Based Early Detection of Parkinson's Disease"

test_description = (
    "A mobile application that uses machine learning "
    "and wearable sensor data to detect early signs "
    "of Parkinson's disease and monitor patient symptoms."
)

bm25_results = bm25_search(
    title=test_title,
    description=test_description,
    top_k=5
)

display(
    bm25_results[
        ["rank", "project_id", "title", "bm25_score"]
    ].round(4)
)

,rank,project_id,title,bm25_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,48.7279
1,2,P005,Thabat – ثبات,30.5957
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,17.6335
3,4,P026,Potato Guard,15.9575
4,5,P083,Remote Delivery of Healthcare Services for Man...,13.2784


## 7. Load Semantic Retrieval Model

Load the same pretrained multilingual Sentence Transformer used in the standalone Semantic Search implementation.

In [ ]:
# Load semantic model
model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(model_name, device=device)

print("Semantic model loaded successfully!")
print("Device:", device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic model loaded successfully!
Device: cuda
Embedding dimension: 768


/tmp/ipykernel_1632/3673033430.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 8. Token-Based Project Chunking

Split graduation project descriptions into overlapping token-safe chunks to preserve information beyond the model's 128-token limit.

In [ ]:
# Configure chunking
max_tokens = model.max_seq_length
special_tokens = model.tokenizer.num_special_tokens_to_add(pair=False)

chunk_size = max_tokens - special_tokens
chunk_overlap = 20
chunk_step = chunk_size - chunk_overlap

assert chunk_step > 0

# Split text into token-safe chunks
def safe_chunk_text(text, overlap=20):
    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    chunks = []
    start = 0

    while start < len(token_ids):
        end = min(
            start + model.max_seq_length - special_tokens,
            len(token_ids)
        )

        while end > start:
            chunk = model.tokenizer.decode(
                token_ids[start:end],
                skip_special_tokens=True
            )

            length = len(model.tokenizer.encode(
                chunk,
                add_special_tokens=True,
                truncation=False
            ))

            if length <= model.max_seq_length:
                break

            end -= 1

        if end == start:
            raise ValueError("Unable to create a valid chunk.")

        chunks.append(chunk)

        if end == len(token_ids):
            break

        start = max(start + 1, end - overlap)

    return chunks


# Chunk all project descriptions
project_chunk_texts = [
    safe_chunk_text(text)
    for text in project_texts
]

all_chunk_texts = [
    chunk
    for chunks in project_chunk_texts
    for chunk in chunks
]

# Validate chunk lengths
assert all(
    len(model.tokenizer.encode(
        chunk,
        add_special_tokens=True,
        truncation=False
    )) <= max_tokens
    for chunk in all_chunk_texts
)

print("Project chunking completed successfully!")
print("Total projects:", len(project_chunk_texts))
print("Total chunks:", len(all_chunk_texts))
print("Average chunks per project:",
      round(np.mean([len(c) for c in project_chunk_texts]), 2))
print("Maximum tokens:", max_tokens)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (208 > 128). Running this sequence through the model will result in indexing errors


Project chunking completed successfully!
Total projects: 103
Total chunks: 298
Average chunks per project: 2.89
Maximum tokens: 128


## 8.1. Generate and Aggregate Chunk Embeddings

In [ ]:
# Generate embeddings for all project chunks
chunk_embeddings = model.encode(
    all_chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Aggregate chunk embeddings per project
chunked_project_embeddings = []

start = 0

for chunks in project_chunk_texts:
    count = len(chunks)

    project_vectors = chunk_embeddings[start:start + count]
    project_embedding = project_vectors.mean(axis=0)

    project_embedding /= np.linalg.norm(project_embedding)

    chunked_project_embeddings.append(project_embedding)
    start += count

chunked_project_embeddings = np.vstack(chunked_project_embeddings)

# Validate embeddings
assert start == len(chunk_embeddings)
assert chunked_project_embeddings.shape == (103, 768)
assert np.isfinite(chunked_project_embeddings).all()

print("Chunked project embeddings generated successfully!")
print("Embeddings shape:", chunked_project_embeddings.shape)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Chunked project embeddings generated successfully!
Embeddings shape: (103, 768)


## 9. Semantic Retrieval Function

Implement semantic retrieval using normalized Sentence Transformer embeddings and Cosine Similarity.

In [ ]:

def semantic_search(title, description, top_k=5):
    # Validate input
    if not title.strip() or not description.strip():
        raise ValueError("Title and description must not be empty.")

    if not 1 <= top_k <= len(hybrid_df):
        raise ValueError("Invalid top_k value.")

    # Prepare and chunk query
    query_text = f"{title.strip()}. {description.strip()}"
    query_chunks = safe_chunk_text(query_text)

    # Generate query chunk embeddings
    query_vectors = model.encode(
        query_chunks,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Aggregate query embeddings
    query_embedding = query_vectors.mean(axis=0, keepdims=True)
    query_embedding /= np.linalg.norm(
        query_embedding, axis=1, keepdims=True
    )

    # Calculate cosine similarity
    scores = cosine_similarity(
        query_embedding,
        chunked_project_embeddings
    )[0]

    # Retrieve Top-K
    top_indices = np.argsort(-scores)[:top_k]

    results = hybrid_df.iloc[top_indices][
        ["project_id", "title", "abstract", "department", "source_page"]
    ].copy()

    results.insert(0, "rank", range(1, len(results) + 1))
    results["similarity_score"] = scores[top_indices]

    return results.reset_index(drop=True)


print("Chunked semantic retrieval is ready!")


Chunked semantic retrieval is ready!


## 10. Reciprocal Rank Fusion (RRF)

Combine BM25 and semantic retrieval rankings using Reciprocal Rank Fusion to produce a unified hybrid ranking.

In [ ]:

# ============================================================
# 10. HYBRID RETRIEVAL WITH RRF
# ============================================================

def hybrid_search(title, description, top_k=5, rrf_k=60):

    if not title.strip() or not description.strip():
        raise ValueError(
            "Title and description must not be empty."
        )

    if not 1 <= top_k <= len(hybrid_df):
        raise ValueError("Invalid top_k.")

    if rrf_k <= 0:
        raise ValueError("rrf_k must be positive.")

    # 1. Semantic retrieval uses original language
    semantic_results = semantic_search(
        title,
        description,
        top_k=len(hybrid_df)
    )

    semantic_ranks = dict(zip(
        semantic_results["project_id"],
        semantic_results["rank"]
    ))

    # 2. BM25 performs Arabic translation internally
    bm25_results = bm25_search(
        title,
        description,
        top_k=len(hybrid_df)
    )

    bm25_ranks = dict(zip(
        bm25_results["project_id"],
        bm25_results["rank"]
    ))

    # 3. Prepare final project-level ranking
    results = hybrid_df[
        [
            "project_id",
            "title",
            "abstract",
            "department",
            "source_page"
        ]
    ].copy()

    results["semantic_rank"] = (
        results["project_id"].map(semantic_ranks)
    )

    results["bm25_rank"] = (
        results["project_id"].map(bm25_ranks)
    )

    # 4. Reciprocal Rank Fusion
    semantic_rrf = (
        1.0 / (rrf_k + results["semantic_rank"])
    )

    bm25_rrf = (
        1.0 / (rrf_k + results["bm25_rank"])
    ).fillna(0.0)

    results["rrf_score"] = semantic_rrf + bm25_rrf

    results["retrieval_mode"] = (
        "hybrid"
        if len(bm25_results) > 0
        else "semantic_fallback"
    )

    # 5. Rank projects
    results = results.sort_values(
        ["rrf_score", "project_id"],
        ascending=[False, True]
    ).head(top_k).copy()

    results.insert(
        0, "rank", range(1, len(results) + 1)
    )

    return results.reset_index(drop=True)


print("Hybrid search updated successfully!")


Hybrid search updated successfully!


## 11. Hybrid Search Testing

Evaluate the hybrid retrieval pipeline using the same English project idea tested with BM25 and Semantic Search.

In [ ]:
# Test hybrid retrieval
hybrid_results = hybrid_search(
    title=test_title,
    description=test_description,
    top_k=5
)

display(
    hybrid_results[
        [
            "rank",
            "project_id",
            "title",
            "bm25_rank",
            "semantic_rank",
            "rrf_score"
        ]
    ].round(4)
)

,rank,project_id,title,bm25_rank,semantic_rank,rrf_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,1.0,1,0.0328
1,2,P005,Thabat – ثبات,2.0,3,0.0320
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,3.0,2,0.0320
3,4,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,7.0,4,0.0306
4,5,P082,Predicting Diabetes Patients Based On Machine ...,6.0,5,0.0305


## 12. Arabic Hybrid Search Testing

Test the hybrid retrieval pipeline using an Arabic project idea to examine cross-lingual retrieval behavior.

In [ ]:
# Test Arabic query
arabic_title = "الكشف المبكر عن مرض باركنسون باستخدام الذكاء الاصطناعي"

arabic_description = (
    "تطبيق يستخدم تعلم الآلة وبيانات أجهزة الاستشعار "
    "القابلة للارتداء للكشف المبكر عن مرض باركنسون "
    "ومراقبة أعراض المرض لدى المرضى."
)

arabic_hybrid_results = hybrid_search(
    title=arabic_title,
    description=arabic_description,
    top_k=5
)

display(
    arabic_hybrid_results[
        [
            "rank",
            "project_id",
            "title",
            "bm25_rank",
            "semantic_rank",
            "rrf_score"
        ]
    ].round(4)
)

,rank,project_id,title,bm25_rank,semantic_rank,rrf_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,1.0,2,0.0325
1,2,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,4.0,3,0.0315
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,3.0,4,0.0315
3,4,P082,Predicting Diabetes Patients Based On Machine ...,8.0,1,0.0311
4,5,P103,Fileless Malware Detection Using Deep Learning...,9.0,6,0.0296


## 13. Arabic BM25 Diagnostic Analysis

Examine BM25 score coverage for Arabic queries to identify potential cross-lingual retrieval limitations.

In [ ]:

# ============================================================
# 13. ARABIC BM25 DIAGNOSTIC AFTER TRANSLATION
# ============================================================

arabic_query = f"{arabic_title}. {arabic_description}"

# Translate Arabic into English before BM25 scoring
translated_query = translate_query_for_bm25(arabic_query)

arabic_bm25_scores = bm25.get_scores(
    tokenize(translated_query)
)

print("ARABIC BM25 DIAGNOSTICS")
print("=" * 55)

print("Original query:")
print(arabic_query)

print("\nTranslated query:")
print(translated_query)

print("\nTotal projects:", len(arabic_bm25_scores))
print(
    "Projects with positive scores:",
    int(np.sum(arabic_bm25_scores > 0))
)
print(
    "Projects with zero scores:",
    int(np.sum(arabic_bm25_scores == 0))
)

# Inspect actual BM25 contributions to hybrid results
diagnostic = arabic_hybrid_results[
    [
        "rank",
        "project_id",
        "title",
        "bm25_rank",
        "semantic_rank",
        "rrf_score"
    ]
].copy()

score_map = dict(
    zip(hybrid_df["project_id"], arabic_bm25_scores)
)

diagnostic["bm25_score"] = (
    diagnostic["project_id"].map(score_map)
)

display(diagnostic.round(4))

print("\nRetrieval mode:",
      arabic_hybrid_results["retrieval_mode"].iloc[0])


ARABIC BM25 DIAGNOSTICS
Original query:
الكشف المبكر عن مرض باركنسون باستخدام الذكاء الاصطناعي. تطبيق يستخدم تعلم الآلة وبيانات أجهزة الاستشعار القابلة للارتداء للكشف المبكر عن مرض باركنسون ومراقبة أعراض المرض لدى المرضى.

Translated query:
Early detection of Parkinson's disease using artificial intelligence. Application uses machine learning and wearable sensor data for early detection of Parkinson's disease and surveillance of patients'symptoms.

Total projects: 103
Projects with positive scores: 92
Projects with zero scores: 11


,rank,project_id,title,bm25_rank,semantic_rank,rrf_score,bm25_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,1.0,2,0.0325,49.0433
1,2,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,4.0,3,0.0315,15.5958
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,3.0,4,0.0315,18.9026
3,4,P082,Predicting Diabetes Patients Based On Machine ...,8.0,1,0.0311,12.5932
4,5,P103,Fileless Malware Detection Using Deep Learning...,9.0,6,0.0296,11.2308



Retrieval mode: hybrid


In [ ]:
# Re-test Arabic hybrid retrieval
arabic_hybrid_results_fixed = hybrid_search(
    title=arabic_title,
    description=arabic_description,
    top_k=5
)

display(
    arabic_hybrid_results_fixed[
        [
            "rank",
            "project_id",
            "title",
            "bm25_rank",
            "semantic_rank",
            "rrf_score"
        ]
    ].round(4)
)

,rank,project_id,title,bm25_rank,semantic_rank,rrf_score
0,1,P025,Early Detection and Monitoring of Parkinson’s ...,1.0,2,0.0325
1,2,P032,دُرَّة DURRA – Ovarian Cancer Diagnosis Using ...,4.0,3,0.0315
2,3,P080,A Smart Mobile Application For Skin Cancer Det...,3.0,4,0.0315
3,4,P082,Predicting Diabetes Patients Based On Machine ...,8.0,1,0.0311
4,5,P103,Fileless Malware Detection Using Deep Learning...,9.0,6,0.0296


## 14. Hybrid Retrieval Validation

Validate hybrid retrieval rankings, result uniqueness, and semantic fallback behavior for Arabic queries.

In [ ]:
# Re-test English hybrid retrieval
english_hybrid_results = hybrid_search(
    test_title, test_description, top_k=5
)

# Validate retrieval outputs
for results in [english_hybrid_results, arabic_hybrid_results_fixed]:
    assert len(results) == 5
    assert results["project_id"].is_unique
    assert results["rank"].tolist() == [1, 2, 3, 4, 5]
    assert results["rrf_score"].is_monotonic_decreasing
    assert np.isfinite(results["rrf_score"]).all()

# Validate Arabic semantic fallback
arabic_semantic_results = semantic_search(
    arabic_title, arabic_description, top_k=5
)


assert len(arabic_hybrid_results_fixed) == 5
assert arabic_hybrid_results_fixed["project_id"].is_unique
assert arabic_hybrid_results_fixed["rrf_score"].is_monotonic_decreasing

print(
    "Arabic retrieval mode:",
    arabic_hybrid_results_fixed["retrieval_mode"].iloc[0]
)

if arabic_hybrid_results_fixed["retrieval_mode"].iloc[0] == "semantic_fallback":
    print("BM25 found no positive matches; semantic fallback applied.")
else:
    print("BM25 and semantic rankings combined using RRF.")


Arabic retrieval mode: hybrid
BM25 and semantic rankings combined using RRF.


## 15. Save Hybrid Retrieval Results

Export English and Arabic hybrid retrieval results for reproducibility and future evaluation.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path("/content/origin_hybrid_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save retrieval results
english_hybrid_results.to_csv(
    output_dir / "hybrid_english_results.csv",
    index=False,
    encoding="utf-8-sig"
)

arabic_hybrid_results_fixed.to_csv(
    output_dir / "hybrid_arabic_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Hybrid retrieval results saved successfully!")

for file in sorted(output_dir.iterdir()):
    print(file.name)

Hybrid retrieval results saved successfully!
hybrid_arabic_results.csv
hybrid_english_results.csv


In [ ]:
import shutil
from google.colab import files

# Create ZIP archive
archive_path = shutil.make_archive(
    "/content/origin_hybrid_outputs",
    "zip",
    output_dir
)

files.download(archive_path)

print("Hybrid search outputs downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hybrid search outputs downloaded successfully!



## 16. Batch Hybrid Retrieval on Development Queries

Evaluate the updated hybrid retrieval system on the 40 development queries using the same evaluation dataset as the semantic and TF-IDF approaches.


In [ ]:

# Load development evaluation queries
eval_url = (
    "https://raw.githubusercontent.com/"
    "esrrahafase-crypto/Origin/main/"
    "data/evaluation/evaluation_queries.csv"
)

eval_df = pd.read_csv(eval_url, encoding="utf-8-sig")

dev_queries = (
    eval_df[eval_df["split"] == "dev"]
    .copy()
    .reset_index(drop=True)
)

assert len(dev_queries) == 40
assert dev_queries["query_id"].is_unique

# Run hybrid retrieval for all development queries
all_hybrid_results = []

for _, query in dev_queries.iterrows():
    results = hybrid_search(
        title=query["query_title"],
        description=query["query_description"],
        top_k=5
    )

    results.insert(0, "query_id", query["query_id"])
    results.insert(1, "language", query["language"])

    all_hybrid_results.append(results)

hybrid_dev_results = pd.concat(
    all_hybrid_results,
    ignore_index=True
)

# Validate batch results
assert hybrid_dev_results["query_id"].nunique() == 40
assert len(hybrid_dev_results) == 200
assert hybrid_dev_results.groupby("query_id").size().eq(5).all()

print("Batch hybrid retrieval completed successfully!")
print("Development queries:", len(dev_queries))
print("Total retrieved results:", len(hybrid_dev_results))
print("Arabic queries:", (dev_queries["language"] == "ar").sum())
print("English queries:", (dev_queries["language"] == "en").sum())

display(hybrid_dev_results.head(10))


[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Batch hybrid retrieval completed successfully!
Development queries: 40
Total retrieved results: 200
Arabic queries: 20
English queries: 20


,query_id,language,rank,project_id,title,abstract,department,source_page,semantic_rank,bm25_rank,rrf_score,retrieval_mode
0,Q003,en,1,P059,Blood Bank Inventory System - Artificial Intel...,Healthcare institutions are usually challenged...,information systems,65,1,1.0,0.032787,hybrid
1,Q003,en,2,P082,Predicting Diabetes Patients Based On Machine ...,Our graduation project applied machine learnin...,Information Technology,90,5,6.0,0.030536,hybrid
2,Q003,en,3,P070,Mama’s Hope,Preeclampsia is a life-threatening pregnancy c...,Data Science and Analysis,77,4,10.0,0.029911,hybrid
3,Q003,en,4,P038,UniFix— Predictive Maintenance,Predictive maintenance is a modern approach to...,Artificial Intelligence,43,9,5.0,0.029877,hybrid
4,Q003,en,5,P016,PulseXPert-Health Disorder Detection,"The aim of this graduation project, PulseXPert...",Computer Science,20,14,2.0,0.029643,hybrid
5,Q004,en,1,P074,بصيرة Baseera: An AI-Based Sound Recognition a...,Individuals with hearing impairments often enc...,Data Science and Analysis,81,1,1.0,0.032787,hybrid
6,Q004,en,2,P036,رَفد RafdSign: Audio-to-Sign Language Translator,Individuals with hearing impairments face sign...,Artificial Intelligence,41,3,6.0,0.031025,hybrid
7,Q004,en,3,P003,واصف Mobile Application for Deaf and Mute indi...,"Our System, Wasif, offers a communication appl...",Computer Science,7,2,8.0,0.030835,hybrid
8,Q004,en,4,P069,Bridging Communication Gaps: A system for Arab...,Sign language plays a critical role in communi...,Data Science and Analysis,76,4,10.0,0.029911,hybrid
9,Q004,en,5,P085,Driver drowsiness detection system,Drowsy driving is one of the major causes of r...,Information Technology,93,5,12.0,0.029274,hybrid



### 16.1. Reference-Based Retrieval Evaluation

Evaluate Hybrid Search using human-selected reference project IDs. Calculate Reference Hit@5 and MRR@5 separately for Arabic and English development queries.


In [ ]:

# Evaluate hybrid retrieval against human-selected references
evaluation_records = []

for _, query in dev_queries.iterrows():
    refs = query["reference_project_ids"]

    # Skip queries without reference project IDs
    if pd.isna(refs) or not str(refs).strip():
        continue

    reference_ids = {
        ref.strip()
        for ref in str(refs).split(";")
        if ref.strip()
    }

    query_results = hybrid_dev_results[
        hybrid_dev_results["query_id"] == query["query_id"]
    ].sort_values("rank")

    # Find ranks of retrieved reference projects
    matched_ranks = query_results.loc[
        query_results["project_id"].isin(reference_ids),
        "rank"
    ].tolist()

    first_rank = min(matched_ranks) if matched_ranks else None

    evaluation_records.append({
        "query_id": query["query_id"],
        "language": query["language"],
        "reference_hit_at_5": int(first_rank is not None),
        "reference_mrr_at_5": (
            1.0 / first_rank if first_rank is not None else 0.0
        )
    })

hybrid_reference_eval = pd.DataFrame(evaluation_records)

# Calculate metrics by language
hybrid_evaluation_summary = (
    hybrid_reference_eval
    .groupby("language")
    .agg(
        evaluated_queries=("query_id", "count"),
        reference_hit_at_5=("reference_hit_at_5", "mean"),
        reference_mrr_at_5=("reference_mrr_at_5", "mean")
    )
    .reset_index()
)

# Validate evaluation coverage
assert len(hybrid_reference_eval) == 26
assert set(hybrid_evaluation_summary["language"]) == {"ar", "en"}
assert hybrid_evaluation_summary["evaluated_queries"].eq(13).all()

print("Hybrid reference evaluation completed successfully!")
print("Evaluated queries:", len(hybrid_reference_eval))

display(hybrid_evaluation_summary.round(4))


Hybrid reference evaluation completed successfully!
Evaluated queries: 26


,language,evaluated_queries,reference_hit_at_5,reference_mrr_at_5
0,ar,13,0.6923,0.5410
1,en,13,0.9231,0.8462



### 16.2. Arabic Query-Level Error Analysis

Identify Arabic development queries where Hybrid Search failed to retrieve human-selected reference projects within the top five results.


In [ ]:

# Identify Arabic queries missed by Hybrid Search
arabic_eval = hybrid_reference_eval[
    hybrid_reference_eval["language"] == "ar"
].copy()

missed_queries = arabic_eval[
    arabic_eval["reference_hit_at_5"] == 0
]

# Add query details and human-selected references
missed_details = missed_queries.merge(
    dev_queries[
        [
            "query_id",
            "query_title",
            "reference_project_ids"
        ]
    ],
    on="query_id",
    how="left"
)

print("Arabic queries evaluated:", len(arabic_eval))
print("Arabic queries missed by Hybrid:", len(missed_details))

display(
    missed_details[
        [
            "query_id",
            "query_title",
            "reference_project_ids",
            "reference_hit_at_5",
            "reference_mrr_at_5"
        ]
    ]
)


Arabic queries evaluated: 13
Arabic queries missed by Hybrid: 4


,query_id,query_title,reference_project_ids,reference_hit_at_5,reference_mrr_at_5
0,Q038,مراجع الإجابات المقالية للمعلم,P053,0,0.0
1,Q046,لوحة مشاركة خريجات البرامج,P084;P011,0,0.0
2,Q047,ميزانية إصلاح أجهزة المعامل,P022;P038,0,0.0
3,Q049,تحليل أسباب شكاوى المتجر,P071,0,0.0



### 16.3. Semantic vs. Hybrid Analysis for Missed Arabic Queries

Compare reference-project rankings in standalone chunked Semantic Search and Hybrid Search to identify whether rank fusion caused the missed references.


In [ ]:

# Compare Semantic and Hybrid retrieval for missed Arabic queries
comparison_records = []

for _, query in missed_details.iterrows():
    query_id = query["query_id"]

    query_info = dev_queries.loc[
        dev_queries["query_id"] == query_id
    ].iloc[0]

    reference_ids = {
        ref.strip()
        for ref in str(query_info["reference_project_ids"]).split(";")
        if ref.strip()
    }

    semantic_results = semantic_search(
        query_info["query_title"],
        query_info["query_description"],
        top_k=len(hybrid_df)
    )

    hybrid_results = hybrid_search(
        query_info["query_title"],
        query_info["query_description"],
        top_k=len(hybrid_df)
    )

    semantic_matches = semantic_results[
        semantic_results["project_id"].isin(reference_ids)
    ]

    hybrid_matches = hybrid_results[
        hybrid_results["project_id"].isin(reference_ids)
    ]

    comparison_records.append({
        "query_id": query_id,
        "reference_ids": ", ".join(sorted(reference_ids)),
        "semantic_best_rank": (
            int(semantic_matches["rank"].min())
            if not semantic_matches.empty else None
        ),
        "hybrid_best_rank": (
            int(hybrid_matches["rank"].min())
            if not hybrid_matches.empty else None
        ),
        "semantic_hit_at_5": int(
            not semantic_matches.empty
            and semantic_matches["rank"].min() <= 5
        ),
        "hybrid_hit_at_5": int(
            not hybrid_matches.empty
            and hybrid_matches["rank"].min() <= 5
        )
    })

missed_comparison = pd.DataFrame(comparison_records)

display(missed_comparison)


,query_id,reference_ids,semantic_best_rank,hybrid_best_rank,semantic_hit_at_5,hybrid_hit_at_5
0,Q038,P053,2,6,1,0
1,Q046,"P011, P084",5,12,1,0
2,Q047,"P022, P038",1,42,1,0
3,Q049,P071,1,32,1,0



### 16.4. BM25 Rank Analysis for Missed Arabic Queries

Inspect the BM25 rankings and translated queries to investigate why relevant projects were pushed down after rank fusion.


In [ ]:

# Inspect BM25 ranks for missed Arabic queries
diagnostic_records = []

for _, query in missed_details.iterrows():
    query_info = dev_queries.loc[
        dev_queries["query_id"] == query["query_id"]
    ].iloc[0]

    query_text = (
        f"{query_info['query_title']}. "
        f"{query_info['query_description']}"
    )

    translated_query = translate_query_for_bm25(query_text)

    reference_ids = [
        ref.strip()
        for ref in str(query_info["reference_project_ids"]).split(";")
        if ref.strip()
    ]

    bm25_results = bm25_search(
        query_info["query_title"],
        query_info["query_description"],
        top_k=len(hybrid_df)
    )

    for ref_id in reference_ids:
        match = bm25_results[
            bm25_results["project_id"] == ref_id
        ]

        diagnostic_records.append({
            "query_id": query_info["query_id"],
            "reference_id": ref_id,
            "bm25_rank": (
                int(match["rank"].iloc[0])
                if not match.empty else None
            ),
            "bm25_score": (
                float(match["bm25_score"].iloc[0])
                if not match.empty else 0.0
            ),
            "translated_query": translated_query
        })

bm25_missed_diagnostics = pd.DataFrame(diagnostic_records)

display(
    bm25_missed_diagnostics.round(4)
)


,query_id,reference_id,bm25_rank,bm25_score,translated_query
0,Q038,P053,24.0,6.1223,We propose a platform in which the teacher imp...
1,Q046,P084,63.0,1.8378,Programme coordinators need to know the extent...
2,Q046,P011,29.0,4.8533,Programme coordinators need to know the extent...
3,Q047,P022,NaN,0.0000,We propose a tool that helps the operator comp...
4,Q047,P038,NaN,0.0000,We propose a tool that helps the operator comp...
5,Q049,P071,NaN,0.0000,We suggest a board that receives Arab comments...



### 16.5. Weighted RRF Experiment

Test whether reducing the BM25 contribution improves retrieval for the Arabic queries that were missed by standard RRF.


In [ ]:

# Experiment with weighted RRF without modifying hybrid_search()
def weighted_hybrid_search(title, description, top_k=5,
                           semantic_weight=1.0, bm25_weight=0.25,
                           rrf_k=60):

    semantic_results = semantic_search(
        title, description, top_k=len(hybrid_df)
    )

    bm25_results = bm25_search(
        title, description, top_k=len(hybrid_df)
    )

    semantic_ranks = dict(zip(
        semantic_results["project_id"],
        semantic_results["rank"]
    ))

    bm25_ranks = dict(zip(
        bm25_results["project_id"],
        bm25_results["rank"]
    ))

    results = hybrid_df[["project_id", "title"]].copy()

    results["semantic_rank"] = results["project_id"].map(
        semantic_ranks
    )

    results["bm25_rank"] = results["project_id"].map(
        bm25_ranks
    )

    results["weighted_rrf_score"] = (
        semantic_weight / (rrf_k + results["semantic_rank"])
        + bm25_weight / (rrf_k + results["bm25_rank"]).fillna(float("inf"))
    )

    results = results.sort_values(
        ["weighted_rrf_score", "project_id"],
        ascending=[False, True]
    ).reset_index(drop=True)

    results.insert(0, "rank", range(1, len(results) + 1))

    return results.head(top_k)


# Compare rankings for previously missed Arabic queries
weighted_records = []

for _, query in missed_details.iterrows():
    query_info = dev_queries.loc[
        dev_queries["query_id"] == query["query_id"]
    ].iloc[0]

    references = {
        ref.strip()
        for ref in str(query_info["reference_project_ids"]).split(";")
        if ref.strip()
    }

    weighted_results = weighted_hybrid_search(
        query_info["query_title"],
        query_info["query_description"],
        top_k=len(hybrid_df)
    )

    matches = weighted_results[
        weighted_results["project_id"].isin(references)
    ]

    best_rank = int(matches["rank"].min()) if not matches.empty else None

    weighted_records.append({
        "query_id": query["query_id"],
        "standard_hybrid_rank": int(
            missed_comparison.loc[
                missed_comparison["query_id"] == query["query_id"],
                "hybrid_best_rank"
            ].iloc[0]
        ),
        "weighted_hybrid_rank": best_rank,
        "weighted_hit_at_5": int(
            best_rank is not None and best_rank <= 5
        )
    })

weighted_comparison = pd.DataFrame(weighted_records)

display(weighted_comparison)


,query_id,standard_hybrid_rank,weighted_hybrid_rank,weighted_hit_at_5
0,Q038,6,3,1
1,Q046,12,8,0
2,Q047,42,9,0
3,Q049,32,9,0



### 16.6. Weighted RRF Development Evaluation

Evaluate Weighted RRF on all 40 development queries and compare reference-based retrieval performance with standard RRF and standalone Semantic Search.


In [ ]:

# Run Weighted RRF on all development queries
weighted_records = []

for _, query in dev_queries.iterrows():
    results = weighted_hybrid_search(
        title=query["query_title"],
        description=query["query_description"],
        top_k=5,
        semantic_weight=1.0,
        bm25_weight=0.25
    ).copy()

    results.insert(0, "query_id", query["query_id"])
    results.insert(1, "language", query["language"])
    weighted_records.append(results)

weighted_dev_results = pd.concat(
    weighted_records, ignore_index=True
)

assert len(weighted_dev_results) == 200

# Evaluate human-selected references
evaluation_records = []

for _, query in dev_queries.iterrows():
    refs = query["reference_project_ids"]

    if pd.isna(refs) or not str(refs).strip():
        continue

    reference_ids = {
        ref.strip()
        for ref in str(refs).split(";")
        if ref.strip()
    }

    results = weighted_dev_results[
        weighted_dev_results["query_id"] == query["query_id"]
    ]

    matches = results[
        results["project_id"].isin(reference_ids)
    ]

    first_rank = (
        int(matches["rank"].min())
        if not matches.empty else None
    )

    evaluation_records.append({
        "query_id": query["query_id"],
        "language": query["language"],
        "reference_hit_at_5": int(first_rank is not None),
        "reference_mrr_at_5": (
            1 / first_rank if first_rank is not None else 0.0
        )
    })

weighted_reference_eval = pd.DataFrame(evaluation_records)

weighted_evaluation_summary = (
    weighted_reference_eval
    .groupby("language")
    .agg(
        evaluated_queries=("query_id", "count"),
        reference_hit_at_5=("reference_hit_at_5", "mean"),
        reference_mrr_at_5=("reference_mrr_at_5", "mean")
    )
    .reset_index()
)

assert len(weighted_reference_eval) == 26

print("Weighted RRF evaluation completed!")
print("Total retrieval results:", len(weighted_dev_results))
print("Evaluated reference queries:", len(weighted_reference_eval))

display(weighted_evaluation_summary.round(4))


Weighted RRF evaluation completed!
Total retrieval results: 200
Evaluated reference queries: 26


,language,evaluated_queries,reference_hit_at_5,reference_mrr_at_5
0,ar,13,0.6923,0.5513
1,en,13,0.9231,0.7756



### 16.7. Arabic BM25 Weight Sensitivity Analysis

Compare different BM25 fusion weights on Arabic development queries to examine their effect on reference-based retrieval performance.


In [ ]:

# Evaluate different BM25 weights for Arabic queries
bm25_weights = [0.0, 0.05, 0.1, 0.25, 0.5, 1.0]
sensitivity_records = []

arabic_reference_queries = dev_queries[
    (dev_queries["language"] == "ar")
    & dev_queries["reference_project_ids"].notna()
].copy()

for weight in bm25_weights:
    hits = []
    reciprocal_ranks = []

    for _, query in arabic_reference_queries.iterrows():
        results = weighted_hybrid_search(
            title=query["query_title"],
            description=query["query_description"],
            top_k=5,
            semantic_weight=1.0,
            bm25_weight=weight
        )

        references = {
            ref.strip()
            for ref in str(query["reference_project_ids"]).split(";")
            if ref.strip()
        }

        matches = results[
            results["project_id"].isin(references)
        ]

        first_rank = (
            int(matches["rank"].min())
            if not matches.empty else None
        )

        hits.append(int(first_rank is not None))
        reciprocal_ranks.append(
            1.0 / first_rank if first_rank is not None else 0.0
        )

    sensitivity_records.append({
        "bm25_weight": weight,
        "evaluated_queries": len(hits),
        "reference_hit_at_5": np.mean(hits),
        "reference_mrr_at_5": np.mean(reciprocal_ranks)
    })

arabic_weight_sensitivity = pd.DataFrame(sensitivity_records)

print("Arabic BM25 weight sensitivity analysis completed!")
display(arabic_weight_sensitivity.round(4))


Arabic BM25 weight sensitivity analysis completed!


,bm25_weight,evaluated_queries,reference_hit_at_5,reference_mrr_at_5
0,0.00,13,0.9231,0.6141
1,0.05,13,0.8462,0.5538
2,0.10,13,0.8462,0.5282
3,0.25,13,0.6923,0.5513
4,0.50,13,0.6923,0.5449
5,1.00,13,0.6923,0.5410



### 16.8. Language-Aware Retrieval Experiment

Evaluate a language-aware retrieval strategy that uses chunked Semantic Search for Arabic queries and standard Hybrid Search for English queries.


In [ ]:

# Language-aware retrieval without modifying existing functions
def language_aware_search(title, description, language, top_k=5):
    if language == "ar":
        results = semantic_search(title, description, top_k=top_k)
        results["retrieval_mode"] = "semantic_ar"

    elif language == "en":
        results = hybrid_search(title, description, top_k=top_k)
        results["retrieval_mode"] = "hybrid_en"

    else:
        raise ValueError("Unsupported language.")

    return results


# Evaluate all development queries
language_aware_records = []

for _, query in dev_queries.iterrows():
    results = language_aware_search(
        query["query_title"],
        query["query_description"],
        query["language"],
        top_k=5
    ).copy()

    results.insert(0, "query_id", query["query_id"])
    results.insert(1, "language", query["language"])

    language_aware_records.append(results)

language_aware_dev_results = pd.concat(
    language_aware_records,
    ignore_index=True
)

assert len(language_aware_dev_results) == 200
assert language_aware_dev_results["query_id"].nunique() == 40

print("Language-aware retrieval completed successfully!")
print("Total results:", len(language_aware_dev_results))

display(
    language_aware_dev_results.groupby(
        ["language", "retrieval_mode"]
    ).size()
)


Language-aware retrieval completed successfully!
Total results: 200


,,0
language,retrieval_mode,
ar,semantic_ar,100
en,hybrid_en,100



### 16.9. Language-Aware Retrieval Evaluation

Evaluate the language-aware retrieval strategy using human-selected reference project IDs and compare its performance across Arabic and English development queries.


In [ ]:

# Evaluate language-aware retrieval
evaluation_records = []

for _, query in dev_queries.iterrows():
    refs = query["reference_project_ids"]

    if pd.isna(refs) or not str(refs).strip():
        continue

    reference_ids = {
        ref.strip()
        for ref in str(refs).split(";")
        if ref.strip()
    }

    results = language_aware_dev_results[
        language_aware_dev_results["query_id"] == query["query_id"]
    ].sort_values("rank")

    matched_ranks = results.loc[
        results["project_id"].isin(reference_ids),
        "rank"
    ].tolist()

    first_rank = min(matched_ranks) if matched_ranks else None

    evaluation_records.append({
        "query_id": query["query_id"],
        "language": query["language"],
        "reference_hit_at_5": int(first_rank is not None),
        "reference_mrr_at_5": (
            1.0 / first_rank if first_rank is not None else 0.0
        )
    })

language_aware_reference_eval = pd.DataFrame(evaluation_records)

language_aware_evaluation_summary = (
    language_aware_reference_eval
    .groupby("language")
    .agg(
        evaluated_queries=("query_id", "count"),
        reference_hit_at_5=("reference_hit_at_5", "mean"),
        reference_mrr_at_5=("reference_mrr_at_5", "mean")
    )
    .reset_index()
)

# Validate evaluation coverage
assert len(language_aware_reference_eval) == 26
assert language_aware_evaluation_summary["evaluated_queries"].eq(13).all()

print("Language-aware evaluation completed successfully!")
display(language_aware_evaluation_summary.round(4))


Language-aware evaluation completed successfully!


,language,evaluated_queries,reference_hit_at_5,reference_mrr_at_5
0,ar,13,0.9231,0.6141
1,en,13,0.9231,0.8462



## 17. Conclusions

This notebook investigated and optimized hybrid retrieval for Origin by combining BM25, multilingual Semantic Search, and Reciprocal Rank Fusion (RRF).

### Key Findings

- **BM25 Improvement:** Arabic-to-English query translation was introduced to improve lexical matching between Arabic queries and the primarily English graduation project descriptions.
- **Chunked Semantic Retrieval:** Token-based chunking was integrated into Semantic Search to preserve information beyond the model's 128-token limit.
- **Standard Hybrid Search:** BM25 and Chunked Semantic Search were combined using RRF with `k=60`.
- **Development Evaluation:** Retrieval methods were evaluated on 40 development queries (20 Arabic and 20 English). Reference Hit@5 and MRR@5 were calculated using 26 queries with human-selected reference project IDs.
- **Arabic Error Analysis:** Standard RRF reduced Arabic retrieval performance. Query-level analysis revealed cases where relevant projects ranked highly in Semantic Search but received low or zero BM25 scores.
- **Weighted RRF:** Different BM25 fusion weights were tested. Reducing the BM25 weight improved some individual rankings but did not outperform standalone Semantic Search on Arabic development queries.
- **Language-Aware Retrieval:** A language-dependent strategy was introduced to preserve the strongest observed retrieval performance for each language.

### Development Evaluation Results

| Metric | Chunked Semantic | Standard Hybrid | Weighted Hybrid (0.25) | Language-Aware |
|---|---:|---:|---:|---:|
| Arabic Reference Hit@5 | 0.9231 | 0.6923 | 0.6923 | **0.9231** |
| Arabic Reference MRR@5 | 0.6141 | 0.5410 | 0.5513 | **0.6141** |
| English Reference Hit@5 | 0.8462 | 0.9231 | 0.9231 | **0.9231** |
| English Reference MRR@5 | 0.7500 | 0.8462 | 0.7756 | **0.8462** |

### Selected Retrieval Strategy

Based on development-set experiments, Origin selected the following language-aware retrieval configuration:

- **Arabic queries:** Chunked Semantic Search without BM25 contribution.
- **English queries:** Standard Hybrid Search combining BM25 and Chunked Semantic Search through RRF (`k=60`, equal fusion weights).

This configuration achieved a Reference Hit@5 of **0.9231 for both languages** and the strongest observed combined development performance among the tested configurations.

### Limitations and Next Steps

- The selected configuration was developed and compared using the same development dataset; therefore, its performance on unseen queries remains unverified.
- Human-selected reference project IDs may not represent every relevant project in the corpus.
- Further evaluation on the reserved final test set is required after freezing the retrieval configuration.
- The selected retrieval pipeline will subsequently support Origin's Retrieval-Augmented Generation (RAG) component, where an LLM will generate grounded suggestions for improving proposed graduation project ideas.
- RAG-generated suggestions will require separate evaluation for relevance, faithfulness, and practical usefulness.

Overall, the experiments demonstrated that a single fusion strategy was not optimal for both languages. Language-aware retrieval was selected as Origin's development-stage solution, pending independent final-test evaluation.
